# Main Workflow for Investigating UCRB Drought 1998-2022 <br>
## NOTEBOOK 1: Preprocessing Data

The workflow for Nagamoto et al., 2026 for investing UCRB Drought follows 2 main Jupyter Notebooks. The first is for preprocessing raw data, which outputs saved data files that will be used in subsequent notebooks. The second is for running drought impacts code and for plotting. 

This is the **first** notebook, for preprocessing and saving data files. You only need to run this once and then can use the saved datafiles.

## Load Packages and configure notebook

In [1]:
### load packages
import warnings
import preprocessing

warnings.simplefilter('ignore')
%matplotlib inline

In [2]:
### Configuration 
var_names   = ['RDC', 'WT', 'SC']
MET_vars    = ['precip', 'temp']
start_year  = 1998
end_year    = 2022
huc         = '14'
prefix      = 'RDC-WT-SC'
start_date  = '1950-01-01'
end_date    = '2022-12-31'

## Load all input data

In [3]:
### Load all inputs 
dirs, raw_met_data, gagesii, gage_loc, augmentedWT_data, RMSE_WT_data = \
    preprocessing.load_all_inputs()

## Run Preprocessing

In [4]:
# load raw hydrologic variables
data, metadata = preprocessing.load_usgs_basin3d_data(huc, prefix, start_date, end_date, dirs['rdc_wt_sc_raw_dir'])

Returning the data and metadata extracted by the BASIN3D data loader.
------------------------------------------


In [5]:
# process hydrologic variables
list_data_dfs, list_metadata_dfs = preprocessing.make_var_dfs(data, metadata, start_date, end_date, augmentedWT_data, RMSE_WT_data)

Processing RDC data...
Processing WT data...
Finished combining augmented WT data with original.
Finished adding augmented WT data.
Processing SC data...
Finished processing all variables:
- Separated data and metadata by variable
- Extracted daily Mean and calculated Min/Max average
- Checked for ICE and -9999 values
- Regrouped to water years
------------------------------------------


In [6]:
# get availability of hydrologic variables
list_avail_dfs, list_peravail_dfs = preprocessing.apply_criteria_get_avail(list_data_dfs)

Processing availability for RDC...
Processing availability for WT...
Processing availability for SC...
Finished applying criteria: 10 days per month, 11 months per year
------------------------------------------


In [7]:
# correct the file list based on NAN sites
list_data_filtered_dfs,list_metadata_filtered_dfs,list_avail_filtered_dfs = preprocessing.delete_save_sites(dirs['rdc_wt_sc_data_dir'],list_avail_dfs,list_data_dfs,list_metadata_dfs)

Processing RDC data...
  Removing 181 sites with no availability
  Sites with any availability: 925
  Saved RDC data files
Processing WT data...
  Removing 97 sites with no availability
  Sites with any availability: 114
  Saved WT data files
Processing SC data...
  Removing 85 sites with no availability
  Sites with any availability: 102
  Saved SC data files
Finished processing all variables:
- Removed sites with no availability
- Cleared data from years that do not meet criteria
- All data saved to /Users/emilynagamoto/PycharmProjects/ESSDIVE/ESS_DIVE_6-7-2026/UCRB-drought/OUTPUTS/RDC_WT_SC_data/Water_year/
------------------------------------------


In [8]:
# preprocess MET data
temp_df_index, precip_df_index = preprocessing.split_met_data(raw_met_data, dirs['met_data_dir'])

Finished splitting temp and precip.
- All data saved to /Users/emilynagamoto/PycharmProjects/ESSDIVE/ESS_DIVE_6-7-2026/UCRB-drought/OUTPUTS/MET_data/
------------------------------------------


In [9]:
# get landcover change
overall_nlcd_change = preprocessing.nlcd_processing(dirs['nlcd_raw_dir'], dirs['nlcd_data_dir'], start_year, end_year)

Processing NLCD data for counties in AZ, CO, NM, UT, WY...
Found 3 counties for AZ
Found 32 counties for CO
Found 4 counties for NM
Found 16 counties for UT
Found 7 counties for WY
NLCD change calculated for period 1998 to 2022
NLCD data saved to /Users/emilynagamoto/PycharmProjects/ESSDIVE/ESS_DIVE_6-7-2026/UCRB-drought/OUTPUTS/NLCD_data
------------------------------------------


In [10]:
# calculate PET
pet_data_df = preprocessing.calculate_pet(temp_df_index, gage_loc, dirs['met_data_dir'])

PET calculation completed.
PET data saved to /Users/emilynagamoto/PycharmProjects/ESSDIVE/ESS_DIVE_6-7-2026/UCRB-drought/OUTPUTS/MET_data/pet_calc_thorn.csv
------------------------------------------


In [11]:
# calculate SPEI
ann_spei_wyears, TRUN_ann_spei_wyears = preprocessing.calc_SPEI(gagesii,precip_df_index,pet_data_df,dirs['spei_data_dir'],start_year,end_year)

Calculating SPEI (Standardized Precipitation Evapotranspiration Index)...
Calculated water deficit (P - PET) for 372 basins
Calculated 12-month rolling mean of water deficit
Calculated SPEI for 372 basins
Calculated annual SPEI values for water years 1971 to 2023
Truncated annual SPEI covers water years 1998 to 2022
SPEI data saved to /Users/emilynagamoto/PycharmProjects/ESSDIVE/ESS_DIVE_6-7-2026/UCRB-drought/OUTPUTS/SPEI_data
------------------------------------------


In [12]:
# calculate runoff
runoff_ann = preprocessing.Q_to_ann_runoff(list_data_dfs[0], gagesii, dirs['rdc_wt_sc_data_dir'])

Normalized streamflow by basin area (m^3 / s km^2)
Convert Q (m^3 / s km^2) to daily runoff (mm/year), then sum for the whole year.
- Annual Runoff data saved to /Users/emilynagamoto/PycharmProjects/ESSDIVE/ESS_DIVE_6-7-2026/UCRB-drought/OUTPUTS/RDC_WT_SC_data/
------------------------------------------


In [13]:
# calculate runoff efficiency
runoff_efficiency = preprocessing.get_runoff_efficiency(runoff_ann, precip_df_index, gagesii, dirs['rdc_wt_sc_data_dir'])

prep P_ann
isolate to 1998-2022
drop sites that are all nan
calculate runoff ratio (AKA runoff efficiency)
- Annual Runoff Efficiency data saved to /Users/emilynagamoto/PycharmProjects/ESSDIVE/ESS_DIVE_6-7-2026/UCRB-drought/OUTPUTS/RDC_WT_SC_data/
------------------------------------------


In [14]:
# preprocess the separate reservoir storage files
reservoir_storage = preprocessing.combine_reservoir_data(dirs['reservoir_dir'], dirs['reservoir_data_dir'])

- Reservoir Storage data saved to /Users/emilynagamoto/PycharmProjects/ESSDIVE/ESS_DIVE_6-7-2026/UCRB-drought/OUTPUTS/RESERVOIR_data/
------------------------------------------
